# Dataset Preparation

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

raw = load_dataset("DeepMount00/CulturaViva-ITA", split="train")
split_ds = raw.train_test_split(test_size=0.1, seed=42)

train_ds = split_ds["train"]
test_ds = split_ds["test"]


def build_ir_split(ds, split_name):
    queries = []
    corpus = []
    qrels = []

    for i, row in enumerate(ds):
        qid = f"q_{i}"
        did = f"d_{i}"

        queries.append(
            {
                "id": qid,
                "text": row["prompt"],
            }
        )

        corpus.append(
            {
                "id": did,
                "title": "",
                "text": row["answer"],
            }
        )

        qrels.append(
            {
                "query-id": qid,
                "corpus-id": did,
                "score": 1,
            }
        )

    return (
        Dataset.from_list(queries),
        Dataset.from_list(corpus),
        Dataset.from_list(qrels),
    )


queries_train, corpus_train, qrels_train = build_ir_split(train_ds, "train")
queries_test, corpus_test, qrels_test = build_ir_split(test_ds, "test")

repo_id = "lopozz/CulturaViva-Retrieval"

# push to repo
# DatasetDict({
#     "train": queries_train,
#     "test": queries_test,
# }).push_to_hub(repo_id, config_name="queries")

# # push corpus as one config
# DatasetDict({
#     "train": corpus_train,
#     "test": corpus_test,
# }).push_to_hub(repo_id, config_name="corpus")

# # push qrels as one config
# DatasetDict({
#     "train": qrels_train,
#     "test": qrels_test,
# }).push_to_hub(repo_id, config_name="qrels")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 14.88ba/s]
Processing Files (1 / 1): 100%|██████████| 9.64MB / 9.64MB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.50s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 79.57ba/s]
Processing Files (1 / 1): 100%|██████████| 1.07MB / 1.07MB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.34s/ shards]
No files have been modified since last commit. Skipping to prevent empty commit.
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


CommitInfo(commit_url='https://huggingface.co/datasets/lopozz/CulturaViva-Retrieval/commit/2347b286719a879cc5129cea5c4c5d3fa813dd1b', commit_message='Upload dataset', commit_description='', oid='2347b286719a879cc5129cea5c4c5d3fa813dd1b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/lopozz/CulturaViva-Retrieval', endpoint='https://huggingface.co', repo_type='dataset', repo_id='lopozz/CulturaViva-Retrieval'), pr_revision=None, pr_num=None)

# Train

In [ ]:
from datasets import load_dataset

repo_id = "lopozz/CulturaViva-Retrieval"


def build_pair_dataset(
    repo_id: str, split: str, max_examples: int | None = None
) -> Dataset:
    queries_ds = load_dataset(repo_id, "queries", split=split)
    corpus_ds = load_dataset(repo_id, "corpus", split=split)
    qrels_ds = load_dataset(repo_id, "qrels", split=split)

    query_text_by_id = {row["_id"]: row["text"] for row in queries_ds}
    doc_text_by_id = {row["_id"]: row["text"] for row in corpus_ds}

    pairs = []
    for row in qrels_ds:
        if int(row["score"]) > 0:
            qid = row["query-id"]
            did = row["corpus-id"]

            if qid in query_text_by_id and did in doc_text_by_id:
                pairs.append(
                    {
                        "query": query_text_by_id[qid],
                        "answer": doc_text_by_id[did],
                    }
                )

                if max_examples is not None and len(pairs) >= max_examples:
                    break

    return Dataset.from_list(pairs)


train_dataset = build_pair_dataset(repo_id, "train", max_examples=100)
eval_dataset = build_pair_dataset(repo_id, "test", max_examples=10)

print(train_dataset)
print(eval_dataset)

In [ ]:
import logging

from sentence_transformers import (
    SparseEncoder,
    SparseEncoderModelCardData,
    SparseEncoderTrainer,
    SparseEncoderTrainingArguments,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.losses import (
    SparseMultipleNegativesRankingLoss,
    SpladeLoss,
)
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO
)

# 1. Load a model to finetune with 2. (Optional) model card data
mlm_transformer = Transformer("jhu-clsp/mmBERT-small", transformer_task="fill-mask")
vocab_dim = len(mlm_transformer.tokenizer)

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=vocab_dim,
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)


# 4. Define a loss function
loss = SpladeLoss(
    model=model,
    loss=SparseMultipleNegativesRankingLoss(model=model),
    query_regularizer_weight=0,
    document_regularizer_weight=3e-3,
)

batch_size = 1
# 5. (Optional) Specify training arguments
run_name = "inference-free-splade-mmBERT-small-ita"
args = SparseEncoderTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    learning_rate_mapping={
        r"0\.sub_modules\.query\.0\.weight": 1e-3
    },  # Set a higher learning rate for the SparseStaticEmbedding module (see https://huggingface.co/blog/train-sparse-encoder#inference-free-splade)
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    router_mapping={
        "query": "query",
        "answer": "document",
    },  # Map the column names to the routes
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_steps=200,
    report_to="none",
)

# 6. Create a trainer & train
trainer = SparseEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

# 9. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

# 10. (Optional) Push it to the Hugging Face Hub
# model.push_to_hub(run_name)

In [2]:
from sentence_transformers import SparseEncoder
import torch

model = SparseEncoder("nickprock/splade-bert-base-italian-xxl-uncased-cv")

def show_sparse_terms(text, top_k=30):
    emb = model.encode([text], convert_to_tensor=True)

    # SparseEncoder may return a sparse tensor
    row = emb[0]

    if row.is_sparse:
        row = row.to_dense()

    values, indices = torch.topk(row, k=top_k)

    vocab = model.tokenizer.get_vocab()
    inv_vocab = {v: k for k, v in vocab.items()}

    return [
        (inv_vocab[int(idx)], float(score))
        for idx, score in zip(indices, values)
        if float(score) > 0
    ]

print(show_sparse_terms("effetti collaterali tachipirina nei bambini"))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11614.27it/s]


[('##pi', 0.9616959095001221), ('##rina', 0.4469548761844635), ('tachi', 0.43464091420173645), ('bambini', 0.42675817012786865), ('effetto', 0.011501946486532688)]


# Evaluate

In [2]:
from datasets import load_dataset
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sparse_encoder.evaluation import (
    SparseInformationRetrievalEvaluator,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 1) Load Italian subsets
# params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
# names = ["corpus", "queries", "qrels"]
params = {"path": "mteb/WikipediaRetrievalMultilingual", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
# params = {"path": "mteb/WebFAQRetrieval", "split": "test"}
# names = ["ita-corpus", "ita-queries", "ita-qrels"]
# params = {"path": "mteb/MuPLeR-retrieval", "split": "test"}
# names = ["it-corpus", "it-queries", "it-qrels"]
# params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
# names = ["corpus", "queries", "qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
for row in qrels_ds:
    qid = row["query-id"]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        relevant_docs.setdefault(qid, set()).add(cid)

# 4) Build evaluator
ir_evaluator = SparseInformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    batch_size=4,
    accuracy_at_k=[1],
    write_predictions=True
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"

# CHOOSE ONE OPTION:
# model = SparseEncoder(model_path)

# OR

mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    # query_modules=[
    #     SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    # ], # this doesn't work with nickprock
    query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)

model_name = f'{model_path.split('/')[0]}__{model_path.split('/')[1]}'
# 6) Run evaluation
# results = ir_evaluator(model)
# print(results)

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6303.09it/s]


## Token Retention Rate (TRR)

The **Token Retention Rate (TRR)** is a diagnostic metric for sparse neural retrieval models. It quantifies the proportion of semantically significant tokens from an original input sequence that survive the model's sparsity regularization and receive a positive weight in the final latent representation.

#### Mathematical Formulation

Let $x$ be an input text sequence (a query or a document).

Let $\mathcal{V}$ represent the complete vocabulary of the model's tokenizer.

Let $T(x) \subset \mathcal{V}$ denote the set of unique subword tokens extracted from $x$ by the tokenizer.

Let $\mathcal{U} \subset \mathcal{V}$ denote the set of uninformative tokens, specifically defined as the union of standard stopwords, punctuation marks, and model-specific special tokens (e.g., `[CLS]`, `[SEP]`).

We define the set of **relevant original tokens**, $R(x)$, as the relative complement of $\mathcal{U}$ in $T(x)$:

$$ R(x) = T(x) \setminus \mathcal{U} $$

Let $E(x) \in \mathbb{R}^{|\mathcal{V}|}$ be the sparse vector representation of $x$ produced by the neural encoder. We define $S(x)$ as the support set of this representation, representing all vocabulary terms that the model activated with a weight strictly greater than zero:

$$ S(x) = { v \in \mathcal{V} \mid E(x)_v > 0 } $$

The **Token Retention Rate** for a single sequence $x$ is defined as the intersection of the relevant original tokens and the activated sparse tokens, divided by the total number of relevant original tokens:

$$ \text{TRR}(x) = \frac{| R(x) \cap S(x) |}{| R(x) |} $$

*Note: In the boundary condition where an input text consists entirely of stopwords or punctuation ($|R(x)| = 0$), $\text{TRR}(x)$ is mathematically undefined and conventionally omitted from aggregation.*

#### Corpus-Level Aggregation

When evaluating a model across an entire dataset $X$ (such as a corpus of documents or a set of queries), the metric is typically reported as the macro-averaged expected value:

$$ \text{TRR}(X) = \frac{1}{|X|} \sum_{x \in X} \frac{| R(x) \cap S(x) |}{| R(x) |} $$

#### Interpretation

* **$\text{TRR}(x) \to 1.0$:** Indicates high lexical fidelity. The model preserves the exact terminology of the input sequence, functioning similarly to traditional BM25 but with reweighted terms and added expansions.
* **$\text{TRR}(x) \to 0.0$:** Indicates extreme vocabulary pruning or aggressive semantic translation. The model has discarded the original vocabulary in favor of mapping the concept entirely to distinct expansion terms within the latent space. In practice, an unexpectedly low TRR often signals that the model's sparsity penalty (e.g., the FLOPS regularizer in SPLADE) is too aggressive, resulting in the loss of critical exact-match anchors.

In [3]:
import string

stop_words_it = {
    "a", "abbia", "abbiamo", "abbiano", "abbiate", "ad", "agl", "agli", "ai", 
    "al", "all", "alla", "alle", "allo", "anche", "avemmo", "avendo", "avesse", 
    "avessero", "avessi", "avessimo", "aveste", "avesti", "avete", "aveva", 
    "avevamo", "avevano", "avevate", "avevi", "avevo", "avrai", "avranno", 
    "avrebbe", "avrebbero", "avrei", "avremmo", "avremo", "avreste", "avresti", 
    "avrete", "avrò", "avuta", "avute", "avuti", "avuto", "c", "che", "chi", 
    "ci", "coi", "col", "come", "con", "contro", "cui", "da", "dagl", "dagli", 
    "dai", "dal", "dall", "dalla", "dalle", "dallo", "degl", "degli", "dei", 
    "del", "dell", "della", "delle", "dello", "di", "dov", "dove", "e", "ebbe", 
    "ebbero", "ebbi", "ed", "era", "erano", "eravamo", "eravate", "eri", "ero", 
    "essendo", "faccia", "facciamo", "facciano", "facciate", "faccio", "facemmo", 
    "facendo", "facesse", "facessero", "facessi", "facessimo", "faceste", 
    "facesti", "faceva", "facevamo", "facevano", "facevate", "facevi", "facevo", 
    "fai", "fanno", "farai", "faranno", "farebbe", "farebbero", "farei", 
    "faremmo", "faremo", "fareste", "faresti", "farete", "farò", "fece", 
    "fecero", "feci", "fosse", "fossero", "fossi", "fossimo", "foste", "fosti", 
    "fu", "fui", "fummo", "furono", "gli", "ha", "hai", "hanno", "ho", "i", "il", 
    "in", "io", "l", "la", "le", "lei", "li", "lo", "loro", "lui", "ma", "mi", 
    "mia", "mie", "miei", "mio", "ne", "negl", "negli", "nei", "nel", "nell", 
    "nella", "nelle", "nello", "no", "noi", "non", "nostra", "nostre", "nostri", 
    "nostro", "o", "per", "perché", "più", "quale", "quanta", "quante", "quanti", 
    "quanto", "quella", "quelle", "quelli", "quello", "questa", "queste", 
    "questi", "questo", "sarai", "saranno", "sarebbe", "sarebbero", "sarei", 
    "saremmo", "saremo", "sareste", "saresti", "sarete", "sarò", "se", "sei", 
    "si", "sia", "siamo", "siano", "siate", "siete", "sono", "sta", "stai", 
    "stando", "stanno", "starai", "staranno", "starebbe", "starebbero", "starei", 
    "staremmo", "staremo", "stareste", "staresti", "starete", "starò", "stata", 
    "state", "stati", "stato", "stava", "stavamo", "stavano", "stavate", "stavi", 
    "stavo", "stemmo", "stesse", "stessero", "stessi", "stessimo", "steste", 
    "stesti", "stette", "stettero", "stetti", "sto", "su", "sua", "sue", "sugl", 
    "sugli", "sui", "sul", "sull", "sulla", "sulle", "sullo", "suo", "suoi", "ti", 
    "tra", "tu", "tua", "tue", "tuo", "tuoi", "tutta", "tutte", "tutti", "tutto", 
    "un", "una", "uno", "vi", "voi", "vostra", "vostre", "vostri", "vostro"
}
punctuation = set(string.punctuation)


In [ ]:
def is_relevant_token(token: str) -> bool:
    """Checks if a token is a stopword or punctuation."""
    # Clean the WordPiece '##' prefix for accurate stopword matching
    clean_token = token.lower()
    
    if not clean_token:
        return False
    if clean_token in stop_words_it:
        return False
    if all(char in punctuation for char in clean_token):
        return False
    return True

def extract_retained_tokens(sparse_rep, tokenizer):
    """
    Converts a sparse representation returned by SparseEncoder.encode
    into a set of retained tokenizer tokens.
    Handles dict, sparse tensor, and dense tensor outputs.
    """

    if hasattr(sparse_rep, "detach"):
        sparse_rep = sparse_rep.detach().cpu()

        # Sparse tensor
        if sparse_rep.is_sparse:
            sparse_rep = sparse_rep.coalesce()
            token_ids = sparse_rep.indices()

            # If shape is [vocab_size], indices shape is [1, nnz]
            # If shape is [batch, vocab_size], indices shape is [2, nnz]
            if token_ids.shape[0] == 1:
                token_ids = token_ids[0]
            else:
                token_ids = token_ids[-1]

            token_ids = token_ids.tolist()
            return set(tokenizer.convert_ids_to_tokens(token_ids))

        # Dense tensor
        token_ids = sparse_rep.nonzero(as_tuple=True)[0].tolist()
        return set(tokenizer.convert_ids_to_tokens(token_ids))

    raise TypeError(f"Unsupported sparse representation type: {type(sparse_rep)}")

def calculate_retention_ratio(texts, sparse_representations, tokenizer):
    """Calculates the average ratio of retained relevant tokens."""

    retention_ratios = []

    for text, sparse_rep in zip(texts, sparse_representations):
        orig_tokens = tokenizer.tokenize(text)

        orig_relevant = set(
            tok for tok in orig_tokens
            if is_relevant_token(tok) and tok not in tokenizer.all_special_tokens
        )

        if not orig_relevant:
            continue

        retained_tokens = extract_retained_tokens(sparse_rep, tokenizer)

        if len(retained_tokens) == 0:
            retention_ratios.append(0.0)
            continue

        overlap = orig_relevant.intersection(retained_tokens)
        ratio = len(overlap) / len(orig_relevant)
        retention_ratios.append(ratio)

    return sum(retention_ratios) / len(retention_ratios) if retention_ratios else 0.0

In [10]:
sparse_representations

tensor(indices=tensor([[    0,     0,     0,  ...,  1499,  1499,  1499],
                       [  105,   152,   170,  ..., 28256, 28695, 29090]]),
       values=tensor([1.1411, 0.4557, 0.5362,  ..., 0.2303, 0.5088, 0.3485]),
       device='cuda:0', size=(1500, 32102), nnz=121123, layout=torch.sparse_coo)

In [13]:
# --- RUNNING THE CUSTOM EVALUATION ---
print("\n--- Token Retention Evaluation ---")

# Access the tokenizer from your Transformer module
tokenizer = mlm_transformer.tokenizer
batch_size = 50
# 1. Evaluate Query Retention
query_texts = list(queries.values())
print("Calculating Retention for Queries...")
sparse_representations = model.encode(query_texts, batch_size=batch_size, show_progress_bar=True)
avg_query_retention = calculate_retention_ratio(query_texts, sparse_representations, tokenizer)

# 2. Evaluate Relevant Document Retention
# Fetch the texts of all documents that are considered relevant to at least one query
relevant_doc_ids = set()
for doc_ids in relevant_docs.values():
    relevant_doc_ids.update(doc_ids)

# Filter out doc_ids that might be in qrels but missing from the corpus (sanity check)
relevant_doc_texts = [corpus[doc_id] for doc_id in relevant_doc_ids if doc_id in corpus]
sparse_representations = model.encode(relevant_doc_texts, batch_size=batch_size, show_progress_bar=True)

print("\nCalculating Retention for Relevant Documents...")
# avg_doc_retention = calculate_retention_ratio(relevant_doc_texts, sparse_representations, tokenizer)
print(f"\nFinal Metric - Average Query Token Retention Ratio: {avg_query_retention:.4f}")
# print(f"Final Metric - Average Document Token Retention Ratio: {avg_doc_retention:.4f}")


--- Token Retention Evaluation ---
Calculating Retention for Queries...


Batches: 100%|██████████| 30/30 [00:09<00:00,  3.11it/s]



Calculating Retention for Relevant Documents...

Final Metric - Average Query Token Retention Ratio: 0.8408
Final Metric - Average Document Token Retention Ratio: 0.6651


In [ ]:
# lopozz/CulturaViva-Retrieval
# Final Metric - Average Query Token Retention Ratio: 0.7165
# Final Metric - Average Document Token Retention Ratio: 0.2025

# mteb/MuPLeR-retrieval
# Final Metric - Average Query Token Retention Ratio: 0.8544
# Final Metric - Average Document Token Retention Ratio: 0.5398

# mteb/WebFAQRetrieval
# Final Metric - Average Query Token Retention Ratio: 0.7760
# Final Metric - Average Document Token Retention Ratio: 0.6884

# mteb/WikipediaRetrievalMultilingual
# Final Metric - Average Query Token Retention Ratio: 0.8408
# Final Metric - Average Document Token Retention Ratio: 0.6651



## MTEB Benchmark

### Load Tasks

In [2]:
import yaml
import mteb

with open("../configs/ds/mteg-retrieval-ita.yml", "r") as f:
    config = yaml.safe_load(f)

excluded = set(config["excluded_tasks"])

tasks = mteb.get_tasks(
    languages=config["languages"],
    modalities=config["modalities"],
    task_types=config["task_types"],
    exclusive_language_filter=True,
    eval_splits=["test"],
)

tasks = [t for t in tasks if t.metadata.name not in excluded]
tasks[:1]

[MuPLeRRetrieval(name='MuPLeR-retrieval', languages=['ita'])]

### Performance Table

In [5]:
import mteb

model_names = [
    "mteb/baseline-bm25s",
    "nickprock/splade-bert-base-italian-xxl-uncased-cv",
    "google/embeddinggemma-300m",
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
]
cache = mteb.ResultCache("..")
results = cache.load_results(models=model_names, tasks=tasks, require_model_meta=False)

results.to_dataframe()

model_name,task_name,google/embeddinggemma-300m,mteb/baseline-bm25s,nickprock/splade-bert-base-italian-xxl-uncased-cv,opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1
0,MuPLeR-retrieval,0.84454,0.84084,0.510880,0.551909
1,WebFAQRetrieval,0.79523,0.55918,0.517111,0.554244
2,WikipediaRetrievalMultilingual,0.91761,0.84360,0.799550,0.837200


# Post Analysis

### Single Analysis

In [ ]:
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    # query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
    device='cpu'
)

In [ ]:
sentences = [
    "Per importare sostanze controllate nel corso del 2004, le imprese cui è stato assegnato un contingente devono richiedere alla Commissione, attraverso il sito Internet dell'ODS, una licenza di importazione utilizzando gli appositi moduli. Se i servizi della Commissione constatano che la richiesta è conforme alla quota autorizzata e in linea con i requisiti previsti dal regolamento (CE) n. 2037/2000, viene rilasciata una licenza di importazione. La Commissione si riserva il diritto di non concedere al licenza di importazione se la sostanza da importare non corrisponde alla descrizione o se rischia di non essere destinata all'uso autorizzato o ne è vietata l'importazione ai sensi del regolamento.",
    # "Che sole c'è fuori!",
    # "Ha guidato fino allo stadio.",
    # "The patient complained of severe cephalalgia"
]
embeddings = model.encode_document(sentences)
print(embeddings.shape)
# (3, 30522)

# Get the similarity scores for the embeddings
# similarities = model.similarity(embeddings, embeddings)
# print(similarities)
# tensor([[   32.4323,     5.8528,     0.0258],
#         [    5.8528,    26.6649,     0.0302],
#         [    0.0258,     0.0302,    24.0839]])

# Let's decode our embeddings to be able to interpret them
decoded = model.decode(embeddings, top_k=100)
for decoded, sentence in zip(decoded, sentences):
    print(f"Sentence: {sentence}")
    print(f"Decoded: {decoded}")
    print()

### Model Cross-comparison
Identify retrieval queries different model (BM25 and neural sparse models) performs at different performance levels.

This cell loads the Italian split of the MuPLeR retrieval benchmark from MTEB, builds the
query, corpus, and relevance mappings, then compares per-query retrieval performance across
three models:

- BM25 baseline
- OpenSearch neural sparse multilingual encoder
- Italian SPLADE model

For each model, the code loads saved retrieval predictions, computes binary nDCG@10 for every
query using the qrels as gold relevance labels, and stores both the retrieved document IDs and
their scores. The per-model results are then merged into a single comparison dataframe keyed
by query ID.

The final cell allows to select queries at different levels of performance for the examinel 
models. These examples are useful for qualitative error analysis, especially to inspect cases 
where lexical retrieval differs from neural sparse retrieval.

The resulting `subset` dataframe contains the query text, gold document IDs, retrieved document
IDs and scores for each model, and nDCG@10 values. The last line returns the first ten query 
texts from this subset for downstream analysis, for example with query/document expansion scripts. 
The subset can be further analyzed using `scripts/splade/generate_expansions.py` and 
`streamlits/splade_expansion_app.py`.


In [1]:
from datasets import load_dataset

task_name = 'MuPLeR-retrieval'
params = {"path": f"mteb/{task_name}", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
id2text = {}
queries_df = queries_ds.to_pandas()
for row in qrels_ds:
    qid = row["query-id"]
    qtext = queries_df[queries_df['id']==qid]['text'].iloc[0]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        id2text[qid] = qtext
        relevant_docs.setdefault(qid, set()).add(cid)

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import json
import math
import pandas as pd
from pathlib import Path


def dcg_at_k(relevances, k):
    return sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevances[:k])
    )


def ndcg_at_k(retrieved_ids, gold_ids, k):
    gold_ids = set(str(x) for x in gold_ids)

    if not gold_ids:
        return 0.0

    retrieved_relevances = [
        1 if str(cid) in gold_ids else 0
        for cid in retrieved_ids[:k]
    ]

    dcg = dcg_at_k(retrieved_relevances, k)

    ideal_relevances = [1] * min(len(gold_ids), k)
    idcg = dcg_at_k(ideal_relevances, k)

    return dcg / idcg if idcg > 0 else 0.0


def normalize_model_name(model_name):
    org, name = model_name.split("/")
    return f"{org}__{name}"


def load_predictions(model_name, task_name, base_dir="/home/lpozzi/Git/lm-toolkit/results"):
    model_dir = normalize_model_name(model_name)

    path = Path(base_dir) / model_dir / "prediction_folder" / f"{task_name}_predictions.json"

    with open(path, "r") as f:
        return json.load(f)


def per_query_ndcg(
    retrieval_predictions,
    relevant_docs,
    model_label,
    k=10,
    lang="it",
    split="test",
):
    rows = []

    for qid, results in retrieval_predictions[lang][split].items():
        qid_str = str(qid)

        # Safer than relying on JSON order:
        # sort retrieved docs by score descending
        ranked_results = sorted(
            results.items(),
            key=lambda x: x[1],
            reverse=True,
        )[:k]

        retrieved_ids = [str(cid) for cid, _ in ranked_results]
        retrieved_scores = [round(score, 4) for _, score in ranked_results]

        gold_ids = sorted(str(cid) for cid in relevant_docs.get(qid, relevant_docs.get(qid_str, set())))

        score = ndcg_at_k(
            retrieved_ids=retrieved_ids,
            gold_ids=gold_ids,
            k=k,
        )

        rows.append({
            "qid": qid_str,
            f"{model_label}_ndcg@{k}": score,
            f"{model_label}_retrieved_ids": retrieved_ids,
            f"{model_label}_retrieved_scores": retrieved_scores,
            "gold_ids": gold_ids,
        })

    return pd.DataFrame(rows)


models = {
    "bm25": "mteb/baseline-bm25s",
    "opensearch_sparse": "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1",
    "splade_it": "nickprock/splade-bert-base-italian-xxl-uncased-cv",
}

dfs = []

for label, model_name in models.items():
    preds = load_predictions(model_name, task_name)

    df_model = per_query_ndcg(
        retrieval_predictions=preds,
        relevant_docs=relevant_docs,
        model_label=label,
        k=10,
    )

    dfs.append(df_model)

df_compare = dfs[0]

for df in dfs[1:]:
    df_no_gold = df.drop(columns=["gold_ids"], errors="ignore")

    df_compare = df_compare.merge(
        df_no_gold,
        on="qid",
        how="inner",
    )

df_compare.insert(
    loc=df_compare.columns.get_loc("qid") + 1,
    column="qtext",
    value=df_compare["qid"].map(id2text),
)

df_compare.head()

,qid,qtext,bm25_ndcg@10,bm25_retrieved_ids,bm25_retrieved_scores,gold_ids,opensearch_sparse_ndcg@10,opensearch_sparse_retrieved_ids,opensearch_sparse_retrieved_scores,splade_it_ndcg@10,splade_it_retrieved_ids,splade_it_retrieved_scores
0,ba404e28-77f6-4dc0-8c45-b54c22053c83,Come si estendono le tutele fuori locali comme...,1.0,"[6099, 6954, 197, 8502, 8195, 2504, 5546, 2616...","[13.4167, 6.9343, 6.8391, 6.2509, 6.1587, 6.14...",[6099],0.289065,"[2616, 9938, 7519, 7697, 3214, 6954, 4235, 164...","[0.9381, 0.938, 0.9379, 0.9365, 0.9355, 0.9353...",0.000000,"[2825, 4888, 7463, 8394, 9964, 9970, 9827, 850...","[0.9659, 0.9633, 0.9624, 0.9621, 0.9619, 0.961..."
1,274beb3a-3ded-48f5-893c-b77181f358c9,Perché le imprese sottofinanziano la formazion...,1.0,"[8048, 5675, 8052, 9743, 6738, 2674, 3123, 258...","[10.8445, 7.0972, 6.9736, 6.579, 6.404, 6.2114...",[8048],0.000000,"[5285, 5487, 8054, 68, 1757, 80, 5972, 7561, 5...","[0.9329, 0.9313, 0.9295, 0.9284, 0.9282, 0.928...",0.289065,"[6834, 5538, 2586, 6306, 7460, 8157, 6455, 843...","[0.9607, 0.9589, 0.9583, 0.9572, 0.9563, 0.956..."
2,776d33f1-4283-40cb-9463-d4758a5a1ee7,Chi mantiene il potere di firma congiunta sul ...,1.0,"[4499, 9321, 2870, 7634, 9609, 6224, 5244, 375...","[17.7686, 7.3974, 5.5244, 5.5225, 5.4763, 5.47...",[4499],0.000000,"[7519, 5734, 7825, 5022, 6927, 312, 1647, 6478...","[0.9318, 0.928, 0.925, 0.9246, 0.9245, 0.9235,...",0.000000,"[7512, 4564, 5818, 3051, 2561, 428, 2240, 8702...","[0.9493, 0.9485, 0.948, 0.947, 0.9465, 0.9465,..."
3,eaf58e78-e2b7-43ec-b417-4106d5a6eea3,Quale organismo dovrebbe essere istituzionalme...,1.0,"[8583, 8587, 5627, 8595, 8586, 8591, 6632, 858...","[14.6599, 11.5913, 10.4247, 9.241, 8.8096, 8.7...",[8583],1.000000,"[8583, 5868, 1987, 8586, 3257, 2384, 2385, 668...","[0.9356, 0.9312, 0.9308, 0.9294, 0.9291, 0.928...",0.430677,"[9578, 3951, 7665, 8583, 5769, 1377, 7634, 504...","[0.9605, 0.9601, 0.9587, 0.9585, 0.9567, 0.956..."
4,5c849642-6405-4e6a-86ad-8a103649c485,Chi ha concluso che investitori buyout rafforz...,0.0,"[3370, 4886, 2155, 1707, 8349, 6501, 7745, 672...","[6.5714, 6.3976, 5.7079, 5.2298, 4.9371, 4.888...",[9831],0.000000,"[9938, 4234, 3762, 611, 5888, 4901, 3880, 2616...","[0.9487, 0.9462, 0.946, 0.946, 0.9453, 0.9445,...",0.000000,"[8331, 6720, 8937, 9946, 2872, 8347, 5818, 769...","[0.9639, 0.9589, 0.9568, 0.9565, 0.9564, 0.956..."


In [3]:
subset = df_compare[
    (df_compare["bm25_ndcg@10"] == 1.0)
    & (df_compare["opensearch_sparse_ndcg@10"] < 0.5)
    & (df_compare["splade_it_ndcg@10"] < 0.5)
].copy()

subset["bm25_margin_vs_best_other"] = (
    subset["bm25_ndcg@10"]
    - subset[["opensearch_sparse_ndcg@10", "splade_it_ndcg@10"]].max(axis=1)
)

subset = subset.sort_values(
    by="bm25_margin_vs_best_other",
    ascending=False,
)

subset['qtext'].iloc[:10].tolist()

['Quale organismo fornì una soluzione autonoma di gestione movimenti e poi, nel 2006, ispezionò i controlli merci inter-regime?',
 'Chi mantiene il potere di firma congiunta sul conto fruttifero a doppia firma per assicurare la corretta erogazione?',
 'Quale tipo di regolatore deve riesaminare periodicamente le decisioni di mercato nel quadro, a differenza delle autorità garanti della concorrenza?',
 'Come la capacità addizionale dei concorrenti e volontà dei grandi spedizionieri di rivolgersi altrove spiegarono la disciplina post-operazione sulle rotte?',
 'Quale modello analitico del 1983 è criticato per presumere benefici uniformi agli esercenti e acquirenti/venditori non reattivi nei pagamenti?',
 "Chi era il membro del personale giudiziario dell'UE residente a Oetrange, rappresentato da J. Iturriagagoitia Bassas, che chiedeva risarcimento?",
 'Quale riforma del 17 e 28 febbraio 1986 istituì un capitolo ecologico per colmare la lacuna del Trattato di Roma?',
 'Chi avvertì che un ap

### Qualitative Inspection (FP vs FN)
Once you have a failure ID, this function prints a "Trial" view: what the model liked vs. what it should have liked.

In [ ]:
def qualitative_inspection(qid, df_fail, corpus):
    row = df_fail[df_fail['qid'] == qid].iloc[0]
    
    print(f"🔍 ANALYSIS FOR QUERY: {row['query']}\n")
    print(f"❌ FALSE POSITIVES (Top 3 Model Choices):")
    for cid in row['retrieved_ids'][:3]:
        print(f"  - [ID {cid}]: {corpus[cid][:200]}...")
        
    print(f"\n✅ FALSE NEGATIVES (What the model missed):")
    if not row['gold_ids']:
        print("  - No ground truth defined for this query.")
    for cid in row['gold_ids'][:3]:
        print(f"  - [ID {cid}]: {corpus[cid][:200]}...")

# Usage:
qualitative_inspection("05e24457-4e76-4ce9-8baf-53f988f5cc4c", df_performers, corpus)

🔍 ANALYSIS FOR QUERY: Quale bulbo di regione delimitata è classificato sotto supervisione del produttore ma può essere reimballato fuori da quella regione?

❌ FALSE POSITIVES (Top 3 Model Choices):
  - [ID 7362]: Benché l'allegato delinei una serie di procedure relative a contratti di appalto pre-commerciale che, pur non rientrando — in virtù di clausole di esclusione — nell'ambito di applicazione delle Dirett...
  - [ID 4494]: Per evitare ulteriori danneggiamenti che comporterebbero un calo di quantità di prodotto vendibile e soprattutto un calo di qualità dell'intera partita è determinante eseguire tale lavorazione in temp...
  - [ID 9014]: Nella pianificazione delle TEN-E si dovrebbe assegnare un mandato chiaro all'ENTSO-E e all'ACER nonché definire il ruolo di mediazione dell'UE. Il Libro verde non è sufficientemente esplicito su quest...

✅ FALSE NEGATIVES (What the model missed):
  - [ID 7074]: Le varietà Makói vöröshagyma o Makói hagyma sono classificate, imballate ed etichettat

### SPLADE Expansion Diagnosis
To see why a False Positive was ranked so high, we need to see the activated tokens. This function maps the sparse vector back to the vocabulary.

In [ ]:
import torch
from sentence_transformers.sentence_transformer.modules import Transformer


# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

# 4. Map indices to words using the mlm_transformer tokenizer
VOCAB = {v: k for k, v in mlm_transformer.tokenizer.get_vocab().items()}

def diagnose_expansion(text, model, mode="query", top_n=-1):
    """
    Updated to use SparseEncoder's query and document specific encoding logic.
    """
    # 1. Run inference based on mode
    with torch.no_grad():
        if mode == "query":
            # SparseEncoder.encode_query returns a tensor (batch_size, vocab_size)
            vector = model.encode_query([text], convert_to_sparse_tensor=False)[0]
        else:
            # SparseEncoder.encode_document returns a tensor (batch_size, vocab_size)
            vector = model.encode_document([text], convert_to_sparse_tensor=False)[0]
    
    # 3. Move to CPU and convert to list
    weights = vector.cpu().tolist()
    
    activated = []
    for i, w in enumerate(weights):
        if w > 0: # Small epsilon to avoid float noise
            token = VOCAB.get(i, f"[UNK_{i}]")
            activated.append((token, w))
            
    # 5. Sort by weight descending
    sorted_activated = sorted(activated, key=lambda x: x[1], reverse=True)
    
    if top_n == -1:
        return sorted_activated
    return sorted_activated[:top_n]

def format_expansion(activations):
    """Formats the output of diagnose_expansion into a clean list."""
    for word, weight in activations:
        # Aligns word to the left (15 chars) and weight to 4 decimal places
        print(f"         {word:<15} | {weight:.4f}")

def show_intersection(query_weights, doc_weights, top_k=-1):
    """
    Fixed version: Converts list of tuples to dicts internally to handle 
    the intersection and weight lookups properly.
    """
    # Convert lists of (token, weight) tuples into dictionaries for lookup
    q_dict = dict(query_weights)
    d_dict = dict(doc_weights)
    
    # Find common tokens using set intersection on the dictionary keys
    common_tokens = set(q_dict.keys()) & set(d_dict.keys())
    
    # Build the intersection data
    intersection = {k: (q_dict[k], d_dict[k]) for k in common_tokens}
    
    # Sort by the product (contribution to total score)
    sorted_inter = sorted(intersection.items(), key=lambda x: x[1][0] * x[1][1], reverse=True)
    
    print(f"    Intersection ({len(sorted_inter)} tokens):")
    if not sorted_inter:
        print("      [No Overlap]")
    else:
        # Handle the top_k slicing (if -1, show everything)
        display_list = sorted_inter if top_k == -1 else sorted_inter[:top_k]
        
        for token, (qw, dw) in display_list:
            # We also calculate the product to show the actual impact on retrieval
            product = qw * dw
            print(f"      - {token:<15} | Q: {qw:.4f} * D: {dw:.4f} = Score: {product:.4f}")


def qualitative_expansion_inspection(qid, df_performers, corpus, queries, model):
    row = df_performers[df_performers['qid'] == qid].iloc[0]
    
    print(f"🔍 ANALYSIS FOR QUERY: {row['query']}")
    # query branch is not using SPLADE pooling
    #     token appears in query -> 1.0
    #     token not in query     -> 0.0
    query_expansion = diagnose_expansion(queries[qid], model, mode="query", top_n=-1)
    # format_expansion(query_expansion[:15]) # Show top 15 for query clarity

    # 1. Identify True Positives (Gold IDs that actually appear in Retrieved IDs)
    tps = [cid for cid in row['gold_ids'] if cid in row['retrieved_ids']]
    if tps:
        print(f"\n🚀 TRUE POSITIVES:")
        for cid in tps:
            rank = row['retrieved_ids'].index(cid) + 1
            score = row['retrieved_scores'][rank-1]
            print(f"  - [ID {cid}] | Rank: {rank} | Score: {score:.4f}")
            print(f"    Text: {corpus[cid][:500]}...")
            doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
            show_intersection(query_expansion, doc_expansion)

    # 2. False Positives (Top of the list, but not in gold)
    print(f"\n❌ FALSE POSITIVES (Top 3 Model Choices):")
    for i, cid in enumerate(row['retrieved_ids'][:3]):
        if cid not in row['gold_ids']:
            score = row['retrieved_scores'][i]
            print(f"  - [ID {cid}] | Rank: {i+1} | Score: {score:.4f}")
            print(f"    Text: {corpus[cid][:500]}...")
            doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
            show_intersection(query_expansion, doc_expansion)
        
    # 3. False Negatives (Gold IDs that did NOT make it into the top k)
    cid  = row['gold_ids'][0]
    if cid not in row['retrieved_ids']:
        print(f"\n✅ FALSE NEGATIVES (What the model missed):")
        score = row['gold_scores'][0]
        print(f"  - [ID {cid}] | Score: {score:.4f} (Below Retrieval Threshold)")
        print(f"    Text: {corpus[cid][:500]}...")
        # Fixed: Define doc_expansion BEFORE showing intersection
        doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
        show_intersection(query_expansion, doc_expansion, top_k=10)
    else:
        print(f"\n✅ Zero FALSE NEGATIVES.")



def write_expansion_json(
    texts,
    model,
    model_name,
    output_path,
    mode="query",
    top_n=-1,
    ensure_ascii=False,
):
    output_path = Path(output_path)

    rows = []
    for idx, text in enumerate(texts):
        expansion = diagnose_expansion(
            text=text,
            model=model,
            mode=mode,
            top_n=top_n,
        )

        rows.append(
            {
                "id": idx,
                "text": text,
                "mode": mode,
                "expansion": [
                    {
                        "token": token,
                        "weight": float(weight),
                    }
                    for token, weight in expansion
                ],
            }
        )

    data = {
        model_name: rows
    }

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2)

In [43]:
for target_qid in df_performers['qid'].iloc[:1].tolist():
    qualitative_expansion_inspection(
        qid=target_qid,
        df_performers=df_performers,
        corpus=corpus,
        queries=queries,
        model=model,
    )

🔍 ANALYSIS FOR QUERY: Quale bulbo di regione delimitata è classificato sotto supervisione del produttore ma può essere reimballato fuori da quella regione?

❌ FALSE POSITIVES (Top 3 Model Choices):
  - [ID 7362] | Rank: 1 | Score: 4.7208
    Text: Benché l'allegato delinei una serie di procedure relative a contratti di appalto pre-commerciale che, pur non rientrando — in virtù di clausole di esclusione — nell'ambito di applicazione delle Direttive, sono però conformi al quadro giuridico vigente, esiste pur sempre la possibilità di una violazione, magari inconsapevole, di tale normativa. Il CESE raccomanda dunque ai committenti di esaminare attentamente l'allegato e di seguirne scrupolosamente le raccomandazioni. Nel caso in cui l'amminist...
    Intersection (5 tokens):
      - ##e             | Q: 1.0000 * D: 2.0762 = Score: 2.0762
      - deli            | Q: 1.0000 * D: 1.2698 = Score: 1.2698
      - ##o             | Q: 1.0000 * D: 0.6573 = Score: 0.6573
      - ##to            | Q

In [ ]:
print(queries['ba404e28-77f6-4dc0-8c45-b54c22053c83'])
diagnose_expansion('Il medico ha prescritto un farmaco per ridurre la pressione alta.', model, mode="doc", top_n=-1)

In [7]:
import json
import torch
from pathlib import Path
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# 5) Load sparse model
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

VOCAB = {v: k for k, v in mlm_transformer.tokenizer.get_vocab().items()}

def diagnose_expansion(text, model, mode="query", top_n=-1):
    """
    Updated to use SparseEncoder's query and document specific encoding logic.
    """
    # 1. Run inference based on mode
    with torch.no_grad():
        if mode == "query":
            # SparseEncoder.encode_query returns a tensor (batch_size, vocab_size)
            vector = model.encode_query([text], convert_to_sparse_tensor=False)[0]
        else:
            # SparseEncoder.encode_document returns a tensor (batch_size, vocab_size)
            vector = model.encode_document([text], convert_to_sparse_tensor=False)[0]
    
    # 3. Move to CPU and convert to list
    weights = vector.cpu().tolist()
    
    activated = []
    for i, w in enumerate(weights):
        if w > 0: # Small epsilon to avoid float noise
            token = VOCAB.get(i, f"[UNK_{i}]")
            activated.append((token, w))
            
    # 5. Sort by weight descending
    sorted_activated = sorted(activated, key=lambda x: x[1], reverse=True)
    
    if top_n == -1:
        return sorted_activated
    return sorted_activated[:top_n]

def write_expansion_json(
    texts,
    model,
    model_name,
    output_path,
    mode="query",
    top_n=-1,
    ensure_ascii=False,
):
    output_path = Path(output_path)

    rows = []
    for idx, text in enumerate(texts):
        expansion = diagnose_expansion(
            text=text,
            model=model,
            mode=mode,
            top_n=top_n,
        )

        rows.append(
            {
                "id": idx,
                "text": text,
                "mode": mode,
                "expansion": [
                    {
                        "token": token,
                        "weight": float(weight),
                    }
                    for token, weight in expansion
                ],
            }
        )

    data = {
        model_name: rows
    }

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2)

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    # query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
    device='cpu'
)

sentences = [
    "Il medico ha prescritto un farmaco per ridurre la pressione alta.",
    "Dopo l’incidente, l’auto è stata portata dal meccanico.",
    "Il ragazzo ha comprato un nuovo telefono perché il vecchio si era rotto.",
    "La banca ha approvato il prestito per l’acquisto della casa.",
    "Il cane correva felice nel parco durante una giornata di sole.",
    "La squadra ha perso la partita nonostante una prestazione eccellente.",
    "Non ho detto che Marco abbia rubato il portafoglio.",
    "Questo ristorante è una perla nascosta nel centro storico.",
    "Il professore ha spiegato la teoria della relatività con esempi semplici.",
    "La consegna del pacco è stata rimandata a causa dello sciopero dei trasporti.",
]

sentences = subset['qtext'].iloc[:10].tolist()

write_expansion_json(
    texts=sentences,
    model=model,
    model_name=model_path.split('/')[-1],
    output_path="splade_expansions.json",
    mode="doc",
    top_n=50,
)

In [8]:
import json
from pathlib import Path

import bm25s


sentences = [
    "Il medico ha prescritto un farmaco per ridurre la pressione alta.",
    "Dopo l’incidente, l’auto è stata portata dal meccanico.",
    "Il ragazzo ha comprato un nuovo telefono perché il vecchio si era rotto.",
    "La banca ha approvato il prestito per l’acquisto della casa.",
    "Il cane correva felice nel parco durante una giornata di sole.",
    "La squadra ha perso la partita nonostante una prestazione eccellente.",
    "Non ho detto che Marco abbia rubato il portafoglio.",
    "Questo ristorante è una perla nascosta nel centro storico.",
    "Il professore ha spiegato la teoria della relatività con esempi semplici.",
    "La consegna del pacco è stata rimandata a causa dello sciopero dei trasporti.",
]

sentences = subset['qtext'].iloc[:10].tolist()


def build_bm25(sentences):
    corpus_tokens = bm25s.tokenize(sentences)
    retriever = bm25s.BM25()
    retriever.index(corpus_tokens)
    return retriever


def bm25_doc_expansion(text, doc_id, retriever, n_docs, top_n=-1):
    # Get readable token strings for this document
    tokens = bm25s.tokenize([text], return_ids=False)[0]

    expansion = []

    for token in sorted(set(tokens)):
        # Score this one token as a query against the whole corpus
        query_tokens = bm25s.tokenize(token)
        doc_ids, scores = retriever.retrieve(query_tokens, k=n_docs)

        # Find the score assigned to this document
        score_by_doc = dict(zip(doc_ids[0].tolist(), scores[0].tolist()))
        weight = float(score_by_doc.get(doc_id, 0.0))

        if weight > 0:
            expansion.append((token, weight))

    expansion = sorted(expansion, key=lambda x: x[1], reverse=True)

    if top_n != -1:
        expansion = expansion[:top_n]

    return expansion


def write_expansion_json_bm25(
    sentences,
    output_path,
    model_name="mteb/baseline-bm25s",
    mode="doc",
    top_n=-1,
    ensure_ascii=False,
):
    retriever = build_bm25(sentences)
    n_docs = len(sentences)

    rows = []

    for idx, text in enumerate(sentences):
        expansion = bm25_doc_expansion(
            text=text,
            doc_id=idx,
            retriever=retriever,
            n_docs=n_docs,
            top_n=top_n,
        )

        rows.append(
            {
                "id": idx,
                "text": text,
                "mode": mode,
                "expansion": [
                    {
                        "token": token,
                        "weight": weight,
                    }
                    for token, weight in expansion
                ],
            }
        )

    data = {model_name: rows}

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2)

    return data

In [9]:
data = write_expansion_json_bm25(
    sentences=sentences,
    output_path="bm25_expansions.json",
    top_n=-1,
)